# 🚦 Traffic Sign Recognition
**Dataset:** [Traffic Sign Dataset Classification](https://www.kaggle.com/datasets/ahemateja19bec1025/traffic-sign-dataset-classification) — 58 classes  
**Approach:** Custom CNN → Transfer Learning (EfficientNetB0) → Fine-tuning  
**Goal:** Classify traffic signs from images with high accuracy


## 1. Setup & Data Download

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download ahemateja19bec1025/traffic-sign-dataset-classification

In [ ]:
import zipfile
with zipfile.ZipFile('/content/traffic-sign-dataset-classification.zip', 'r') as zip_ref:
    zip_ref.extractall('/content')
print("✅ Dataset extracted successfully")

## 2. Imports & Configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Activation,
                                      Dropout, Flatten, Dense,
                                      BatchNormalization, GlobalAveragePooling2D)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_preprocess

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

In [ ]:
# Paths
train_path = '/content/traffic_Data/DATA'
test_path  = '/content/traffic_Data/TEST'

NUM_CLASSES = 58
BATCH_SIZE  = 64
SEED        = 42

## 3. Exploratory Data Analysis

In [ ]:
# Load label names
labels_df = pd.read_csv('/content/labels.csv')
print(labels_df.head())
# Build a dict: class_id (int) -> label name
# Adjust column names if yours differ
label_map = dict(zip(labels_df.iloc[:, 0].astype(int), labels_df.iloc[:, 1]))
print(f"\nTotal classes: {len(label_map)}")

In [ ]:
# Count images per class
class_counts = {}
for folder in sorted(os.listdir(train_path), key=lambda x: int(x)):
    class_counts[int(folder)] = len(os.listdir(os.path.join(train_path, folder)))

total_images = sum(class_counts.values())
print(f"Total training images: {total_images}")
print(f"Min images in a class: {min(class_counts.values())}")
print(f"Max images in a class: {max(class_counts.values())}")

In [ ]:
# ── Class distribution bar chart ──
fig, ax = plt.subplots(figsize=(18, 5))

class_ids = list(class_counts.keys())
counts    = list(class_counts.values())
colors    = plt.cm.tab20c(np.linspace(0, 1, NUM_CLASSES))

bars = ax.bar(class_ids, counts, color=colors, edgecolor='white', linewidth=0.5)
ax.set_xlabel('Class ID', fontsize=12)
ax.set_ylabel('Number of Images', fontsize=12)
ax.set_title('📊 Training Image Distribution per Class', fontsize=15, fontweight='bold')
ax.set_xticks(class_ids)
ax.set_xticklabels(class_ids, rotation=90, fontsize=7)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Sample images grid (2 random images per class, first 12 classes) ──
fig, axes = plt.subplots(4, 6, figsize=(16, 11))
fig.suptitle('🖼️ Sample Training Images (one per class)', fontsize=15, fontweight='bold', y=1.01)

show_classes = sorted(class_counts.keys())[:24]
for ax, cls in zip(axes.flatten(), show_classes):
    cls_dir = os.path.join(train_path, str(cls))
    img_file = np.random.choice(os.listdir(cls_dir))
    img = image.load_img(os.path.join(cls_dir, img_file), target_size=(64, 64))
    ax.imshow(img)
    name = label_map.get(cls, f'Class {cls}')
    ax.set_title(f'{cls}: {name[:18]}', fontsize=7, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 4. Data Generators

In [ ]:
# ── Build test DataFrame ──
test_filenames = sorted(os.listdir(test_path))
test_labels    = [str(int(f.split('_')[0])) for f in test_filenames]
test_df = pd.DataFrame({'filename': test_filenames, 'label': test_labels})
print(f"Test samples: {len(test_df)}")
test_df.head()

## 5. Model 1 — Custom CNN (Baseline)

In [ ]:
IMG_SIZE_CNN = 64

cnn_train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    brightness_range=[0.6, 1.4],
    validation_split=0.2
).flow_from_directory(
    train_path, target_size=(IMG_SIZE_CNN, IMG_SIZE_CNN),
    batch_size=BATCH_SIZE, class_mode='categorical',
    subset='training', shuffle=True,
    classes=[str(i) for i in range(NUM_CLASSES)]
)

cnn_val_gen = ImageDataGenerator(
    rescale=1./255, validation_split=0.2
).flow_from_directory(
    train_path, target_size=(IMG_SIZE_CNN, IMG_SIZE_CNN),
    batch_size=BATCH_SIZE, class_mode='categorical',
    subset='validation', shuffle=False,
    classes=[str(i) for i in range(NUM_CLASSES)]
)

cnn_test_gen = ImageDataGenerator(rescale=1./255).flow_from_dataframe(
    dataframe=test_df, directory=test_path,
    x_col='filename', y_col='label',
    target_size=(IMG_SIZE_CNN, IMG_SIZE_CNN),
    batch_size=1, class_mode='categorical', shuffle=False,
    classes=list(cnn_train_gen.class_indices.keys())
)

In [ ]:
def build_cnn(num_classes=NUM_CLASSES):
    model = Sequential([
        # Block 1
        Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(64, 64, 3)),
        BatchNormalization(),
        Conv2D(32, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2, 2),
        Dropout(0.25),

        # Block 2
        Conv2D(64, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(64, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2, 2),
        Dropout(0.25),

        # Block 3
        Conv2D(128, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2, 2),
        Dropout(0.25),

        # Classifier head
        Flatten(),
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    return model

cnn_model = build_cnn()
cnn_model.summary()

In [ ]:
cnn_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_cnn = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
]

history_cnn = cnn_model.fit(
    cnn_train_gen,
    epochs=50,
    validation_data=cnn_val_gen,
    callbacks=callbacks_cnn
)

In [ ]:
def plot_history(history, title='Training History'):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=14, fontweight='bold')

    # Accuracy
    ax1.plot(history.history['accuracy'],     label='Train Accuracy',      color='royalblue',   linewidth=2)
    ax1.plot(history.history['val_accuracy'], label='Val Accuracy',        color='tomato',      linewidth=2, linestyle='--')
    ax1.set_title('Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(alpha=0.3)

    # Loss
    ax2.plot(history.history['loss'],     label='Train Loss',     color='royalblue',   linewidth=2)
    ax2.plot(history.history['val_loss'], label='Val Loss',       color='tomato',      linewidth=2, linestyle='--')
    ax2.set_title('Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_history(history_cnn, title='Custom CNN — Training History')

In [ ]:
# Evaluate CNN
cnn_test_gen.reset()
cnn_preds  = cnn_model.predict(cnn_test_gen, steps=len(cnn_test_gen), verbose=1)
y_pred_cnn = np.argmax(cnn_preds, axis=1)
y_true     = cnn_test_gen.classes

cnn_acc = accuracy_score(y_true, y_pred_cnn)
print(f"\n✅ Custom CNN — Test Accuracy: {cnn_acc * 100:.2f}%")

## 6. Model 2 — Transfer Learning with EfficientNetB0

In [ ]:
IMG_SIZE_EFF = 224   # EfficientNetB0 native size

eff_train_gen = ImageDataGenerator(
    preprocessing_function=eff_preprocess,
    rotation_range=15,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    brightness_range=[0.6, 1.4],
    validation_split=0.2
).flow_from_directory(
    train_path, target_size=(IMG_SIZE_EFF, IMG_SIZE_EFF),
    batch_size=BATCH_SIZE, class_mode='categorical',
    subset='training', shuffle=True,
    classes=[str(i) for i in range(NUM_CLASSES)]
)

eff_val_gen = ImageDataGenerator(
    preprocessing_function=eff_preprocess, validation_split=0.2
).flow_from_directory(
    train_path, target_size=(IMG_SIZE_EFF, IMG_SIZE_EFF),
    batch_size=BATCH_SIZE, class_mode='categorical',
    subset='validation', shuffle=False,
    classes=[str(i) for i in range(NUM_CLASSES)]
)

eff_test_gen = ImageDataGenerator(
    preprocessing_function=eff_preprocess
).flow_from_dataframe(
    dataframe=test_df, directory=test_path,
    x_col='filename', y_col='label',
    target_size=(IMG_SIZE_EFF, IMG_SIZE_EFF),
    batch_size=1, class_mode='categorical', shuffle=False,
    classes=list(eff_train_gen.class_indices.keys())
)

In [ ]:
# ── Phase 1: Freeze base, train head ──
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE_EFF, IMG_SIZE_EFF, 3)
)
base_model.trainable = False

x      = base_model.output
x      = GlobalAveragePooling2D()(x)
x      = Dense(256, activation='relu')(x)
x      = BatchNormalization()(x)
x      = Dropout(0.5)(x)
output = Dense(NUM_CLASSES, activation='softmax')(x)

eff_model = Model(inputs=base_model.input, outputs=output)
eff_model.summary()

In [ ]:
eff_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_eff = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)
]

print("=== Phase 1: Training head (base frozen) ===")
history_eff_p1 = eff_model.fit(
    eff_train_gen,
    epochs=20,
    validation_data=eff_val_gen,
    callbacks=callbacks_eff
)

In [ ]:
plot_history(history_eff_p1, title='EfficientNetB0 — Phase 1 (Frozen Base)')

In [ ]:
# ── Phase 2: Unfreeze last 20 layers and fine-tune ──
for layer in base_model.layers[-20:]:
    layer.trainable = True

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f"Trainable base layers: {trainable_count} / {len(base_model.layers)}")

eff_model.compile(
    optimizer=Adam(learning_rate=1e-5),   # Much lower LR for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\n=== Phase 2: Fine-tuning last 20 layers ===")
history_eff_p2 = eff_model.fit(
    eff_train_gen,
    epochs=20,
    validation_data=eff_val_gen,
    callbacks=callbacks_eff
)

In [ ]:
plot_history(history_eff_p2, title='EfficientNetB0 — Phase 2 (Fine-tuned)')

## 7. Evaluation & Metrics

In [ ]:
# ── Predict on test set ──
eff_test_gen.reset()
eff_preds  = eff_model.predict(eff_test_gen, steps=len(eff_test_gen), verbose=1)
y_pred_eff = np.argmax(eff_preds, axis=1)

eff_acc = accuracy_score(y_true, y_pred_eff)
print(f"\n✅ EfficientNetB0 — Test Accuracy: {eff_acc * 100:.2f}%")
print(f"✅ Custom CNN     — Test Accuracy: {cnn_acc * 100:.2f}%")

In [ ]:
# ── Classification Report ──
class_names = [label_map.get(i, str(i)) for i in range(NUM_CLASSES)]

print("\n📋 Classification Report — EfficientNetB0")
print("=" * 60)
print(classification_report(y_true, y_pred_eff, target_names=class_names))

In [ ]:
# ── Confusion Matrix ──
cm = confusion_matrix(y_true, y_pred_eff)

plt.figure(figsize=(20, 16))
sns.heatmap(
    cm, annot=False, fmt='d',
    cmap='Blues',
    xticklabels=[str(i) for i in range(NUM_CLASSES)],
    yticklabels=[str(i) for i in range(NUM_CLASSES)]
)
plt.title('🔥 Confusion Matrix — EfficientNetB0', fontsize=16, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Per-class accuracy bar chart ──
per_class_acc = cm.diagonal() / cm.sum(axis=1)

fig, ax = plt.subplots(figsize=(18, 5))
colors = ['tomato' if a < 0.8 else 'mediumseagreen' for a in per_class_acc]
ax.bar(range(NUM_CLASSES), per_class_acc * 100, color=colors, edgecolor='white')
ax.axhline(y=90, color='navy', linestyle='--', linewidth=1.5, label='90% threshold')
ax.set_xlabel('Class ID', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('📊 Per-Class Accuracy — EfficientNetB0 (🟥 <80%, 🟩 ≥80%)', fontsize=13, fontweight='bold')
ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels(range(NUM_CLASSES), rotation=90, fontsize=8)
ax.set_ylim(0, 105)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 7d. ROC Curve (Macro & Micro Average)

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import numpy as np
import matplotlib.pyplot as plt

# Binarize true labels for multi-class ROC
y_true_bin = label_binarize(y_true, classes=np.arange(NUM_CLASSES))

# ── Per-class ROC AUC (quiet) ──
fpr_dict, tpr_dict, roc_auc_dict = {}, {}, {}
for i in range(NUM_CLASSES):
    fpr_dict[i], tpr_dict[i], _ = roc_curve(y_true_bin[:, i], eff_preds[:, i])
    roc_auc_dict[i] = auc(fpr_dict[i], tpr_dict[i])

# ── Macro-average ROC ──
all_fpr = np.unique(np.concatenate([fpr_dict[i] for i in range(NUM_CLASSES)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(NUM_CLASSES):
    mean_tpr += np.interp(all_fpr, fpr_dict[i], tpr_dict[i])
mean_tpr /= NUM_CLASSES
macro_auc = auc(all_fpr, mean_tpr)

# ── Micro-average ROC ──
fpr_micro, tpr_micro, _ = roc_curve(y_true_bin.ravel(), eff_preds.ravel())
micro_auc = auc(fpr_micro, tpr_micro)

# ── Plot ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('📈 ROC Curves — EfficientNetB0', fontsize=15, fontweight='bold')

# Left: Macro + Micro summary
ax1.plot(all_fpr, mean_tpr,  color='royalblue', lw=2.5, label=f'Macro-avg  (AUC = {macro_auc:.3f})')
ax1.plot(fpr_micro, tpr_micro, color='tomato',    lw=2.5, label=f'Micro-avg  (AUC = {micro_auc:.3f})')
ax1.plot([0,1],[0,1], 'k--', lw=1, alpha=0.5, label='Random classifier')
ax1.fill_between(all_fpr, mean_tpr, alpha=0.08, color='royalblue')
ax1.set_title('Macro & Micro Average', fontsize=13)
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.legend(loc='lower right', fontsize=11)
ax1.grid(alpha=0.3)

# Right: Top-10 best and worst AUC classes
sorted_aucs = sorted(roc_auc_dict.items(), key=lambda x: x[1])
worst5 = sorted_aucs[:5]
best5  = sorted_aucs[-5:]
colors_map = {**{k: 'tomato'     for k,_ in worst5},
              **{k: 'mediumseagreen' for k,_ in best5}}

for cls_id, _ in worst5 + best5:
    name  = label_map.get(cls_id, str(cls_id))[:20]
    color = colors_map[cls_id]
    auc_v = roc_auc_dict[cls_id]
    ax2.plot(fpr_dict[cls_id], tpr_dict[cls_id], color=color, lw=1.8,
             label=f'{name} ({auc_v:.2f})')

ax2.plot([0,1],[0,1], 'k--', lw=1, alpha=0.5)
ax2.set_title('Top-5 Best 🟢 & Worst 🔴 Classes', fontsize=13)
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate')
ax2.legend(loc='lower right', fontsize=7.5)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n  Macro-avg AUC : {macro_auc:.4f}")
print(f"  Micro-avg AUC : {micro_auc:.4f}")
print(f"  Best  class   : {label_map.get(best5[-1][0], best5[-1][0])} (AUC={best5[-1][1]:.3f})")
print(f"  Worst class   : {label_map.get(worst5[0][0], worst5[0][0])} (AUC={worst5[0][1]:.3f})")


## 9. Model Comparison

In [ ]:
results_summary = pd.DataFrame({
    'Model':         ['Custom CNN (Baseline)', 'EfficientNetB0 (Transfer Learning)'],
    'Test Accuracy': [f"{cnn_acc*100:.2f}%",   f"{eff_acc*100:.2f}%"],
    'Input Size':    ['64×64',                  '224×224'],
    'Params':        [f"{cnn_model.count_params():,}", f"{eff_model.count_params():,}"]
})

results_summary.style.set_properties(**{'text-align': 'center'}).hide(axis='index')

## 10. Visualize Predictions

In [ ]:
def predict_single(img_path, model, img_size, preprocess_fn=None):
    """Load an image, run inference, and return (predicted_class_id, confidence, label_name)."""
    img       = image.load_img(img_path, target_size=(img_size, img_size))
    img_array = image.img_to_array(img)
    if preprocess_fn:
        img_array = preprocess_fn(img_array)
    else:
        img_array = img_array / 255.0
    img_array  = np.expand_dims(img_array, axis=0)
    preds      = model.predict(img_array, verbose=0)
    class_id   = int(np.argmax(preds, axis=1)[0])
    confidence = float(np.max(preds)) * 100
    label      = label_map.get(class_id, f'Class {class_id}')
    return class_id, confidence, label

In [ ]:
# ── Grid: 16 random test images with True vs Predicted labels ──
np.random.seed(SEED)
sample_indices = np.random.choice(len(test_df), 16, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
fig.suptitle('🔍 Predictions on Random Test Images', fontsize=16, fontweight='bold', y=1.01)

for ax, idx in zip(axes.flatten(), sample_indices):
    fname     = test_df.iloc[idx]['filename']
    true_id   = int(test_df.iloc[idx]['label'])
    img_path  = os.path.join(test_path, fname)

    pred_id, confidence, pred_label = predict_single(
        img_path, eff_model, IMG_SIZE_EFF, eff_preprocess
    )
    true_label = label_map.get(true_id, f'Class {true_id}')

    img = image.load_img(img_path)
    ax.imshow(img)
    ax.axis('off')

    correct = pred_id == true_id
    color   = 'green' if correct else 'red'
    icon    = '✅' if correct else '❌'
    ax.set_title(
        f"{icon} True: {true_label[:20]}\nPred: {pred_label[:20]} ({confidence:.1f}%)",
        fontsize=8, color=color, fontweight='bold'
    )

plt.tight_layout()
plt.show()

In [ ]:
# ── Misclassified examples ──
mismatches = np.where(y_true != y_pred_eff)[0]
print(f"Total misclassified: {len(mismatches)} / {len(y_true)} ({len(mismatches)/len(y_true)*100:.1f}%)")

show_n = min(8, len(mismatches))
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('❌ Misclassified Examples', fontsize=14, fontweight='bold')

for ax, idx in zip(axes.flatten(), mismatches[:show_n]):
    fname      = test_df.iloc[idx]['filename']
    true_id    = y_true[idx]
    pred_id    = y_pred_eff[idx]
    confidence = float(eff_preds[idx][pred_id]) * 100

    img = image.load_img(os.path.join(test_path, fname))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(
        f"True: {label_map.get(true_id, true_id)[:18]}\n"
        f"Pred: {label_map.get(pred_id, pred_id)[:18]} ({confidence:.1f}%)",
        fontsize=7.5, color='red'
    )

plt.tight_layout()
plt.show()

## 11. Live Webcam Inference (Local Only)
> ⚠️ This cell requires a **local** Python environment with a webcam.  
> It will **not** run on Google Colab. Download the saved model and run locally.


In [ ]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_preprocess

# ── Config ──
MODEL_PATH   = 'efficientnet_traffic.keras'
IMG_SIZE     = 224
THRESHOLD    = 0.75
FRAME_W, FRAME_H = 640, 480
FONT = cv2.FONT_HERSHEY_SIMPLEX

# ── Load model & label map ──
import pandas as pd, os
_labels_df = pd.read_csv('labels.csv')           # adjust path as needed
_label_map = dict(zip(_labels_df.iloc[:,0].astype(int), _labels_df.iloc[:,1]))

_model = load_model(MODEL_PATH)
print("✅ Model loaded")

def preprocess_frame(frame):
    """Crop centre square, resize, and preprocess for EfficientNetB0."""
    h, w = frame.shape[:2]
    side = min(h, w)
    y0, x0 = (h - side) // 2, (w - side) // 2
    cropped = frame[y0:y0+side, x0:x0+side]
    resized = cv2.resize(cropped, (IMG_SIZE, IMG_SIZE))
    rgb     = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
    arr     = eff_preprocess(rgb.astype(np.float32))
    return np.expand_dims(arr, axis=0)

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH,  FRAME_W)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_H)

print("Press 'q' to quit.")
while True:
    ret, frame = cap.read()
    if not ret:
        break

    inp   = preprocess_frame(frame)
    preds = _model.predict(inp, verbose=0)
    cls   = int(np.argmax(preds))
    conf  = float(np.max(preds))

    # Draw ROI rectangle
    h, w = frame.shape[:2]
    side = min(h, w)
    y0, x0 = (h - side) // 2, (w - side) // 2
    cv2.rectangle(frame, (x0, y0), (x0+side, y0+side), (0,255,0), 2)

    if conf >= THRESHOLD:
        label_text = f"{_label_map.get(cls, cls)}  {conf*100:.1f}%"
        color = (0, 200, 0)
    else:
        label_text = f"Low confidence ({conf*100:.1f}%)"
        color = (0, 100, 255)

    cv2.putText(frame, label_text, (10, 35), FONT, 0.85, color, 2, cv2.LINE_AA)
    cv2.imshow('Traffic Sign Recognition — press q to quit', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


## 12. Save Models

In [ ]:
cnn_model.save('cnn_baseline.keras')
eff_model.save('efficientnet_traffic.keras')
print("✅ Models saved: cnn_baseline.keras, efficientnet_traffic.keras")